# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.950
Hand rule  Precision@50: 0.660


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.950   vs   tree 0.650
Precision@50:  hand rule 0.660   vs   tree 0.640


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Your experiment here
from sklearn.model_selection import GroupShuffleSplit

# --- Experiment 1: does a deeper tree improve in-sample Precision@50, and is it still readable? ---
print("=== Experiment 1: tree depth (in-sample, as in the notebook) ===")
for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42).fit(X, y)
    s = t.predict_proba(X)[:, 1]
    print(f"depth={depth}: leaves={t.get_n_leaves():>2}  P@20={precision_at_k(s, y, 20):.3f}  P@50={precision_at_k(s, y, 50):.3f}")

# --- Experiment 2: client-holdout split (a client's pages never appear in both train and test) ---
print("\n=== Experiment 2: client-holdout train/test split ===")
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
assert set(df.iloc[train_idx]["client_id"]).isdisjoint(df.iloc[test_idx]["client_id"])
print(f"train pages={len(train_idx)}, test pages={len(test_idx)}, test declining rate={y[test_idx].mean():.3f}")

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y[train_idx], y[test_idx]
hand_te = df["hand_rule_score"].iloc[test_idx].values   # fixed thresholds -> just apply to the same test rows

rows = []
for k in (20, 50):
    rows.append((f"hand rule  P@{k}", precision_at_k(hand_te, y_te, k)))
for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
    s = t.predict_proba(X_te)[:, 1]
    for k in (20, 50):
        rows.append((f"tree d={depth} P@{k}", precision_at_k(s, y_te, k)))
for name, v in rows:
    print(f"{name:<18}: {v:.3f}")
print(f"random baseline (declining rate in test): {y_te.mean():.3f}")

# Repeat over several different client splits so one lucky split doesn't decide the story.
print("\n=== Experiment 3: 10 different client-holdout splits (mean +/- std) ===")
import collections
res = collections.defaultdict(list)
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed).split(X, y, groups=df["client_id"]))
    h = df["hand_rule_score"].iloc[te].values
    t = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X.iloc[tr], y[tr])
    s = t.predict_proba(X.iloc[te])[:, 1]
    for k in (20, 50):
        res[f"hand P@{k}"].append(precision_at_k(h, y[te], k))
        res[f"tree(d=2) P@{k}"].append(precision_at_k(s, y[te], k))
for name, vals in res.items():
    print(f"{name:<16}: {np.mean(vals):.3f} +/- {np.std(vals):.3f}")


=== Experiment 1: tree depth (in-sample, as in the notebook) ===
depth=2: leaves= 4  P@20=0.650  P@50=0.640
depth=3: leaves= 8  P@20=0.650  P@50=0.660


depth=4: leaves=16  P@20=0.700  P@50=0.720

=== Experiment 2: client-holdout train/test split ===


train pages=19166, test pages=10834, test declining rate=0.559


hand rule  P@20   : 0.600
hand rule  P@50   : 0.640
tree d=2 P@20     : 0.550
tree d=2 P@50     : 0.620
tree d=3 P@20     : 0.500
tree d=3 P@50     : 0.600
tree d=4 P@20     : 0.500
tree d=4 P@50     : 0.620
random baseline (declining rate in test): 0.559

=== Experiment 3: 10 different client-holdout splits (mean +/- std) ===


hand P@20       : 0.690 +/- 0.180
tree(d=2) P@20  : 0.590 +/- 0.139
hand P@50       : 0.586 +/- 0.123
tree(d=2) P@50  : 0.632 +/- 0.110


**What I observed (directional; my own run, on this sample):**

1. **Depth (in-sample).** Going from depth 2 to 4 raised in-sample Precision@50 from 0.64 to 0.72 (leaves 4 -> 16), but the tree stops being something I can read at a glance, and in-sample gains can be memorisation.
2. **Client-holdout split** (70/30 by `client_id`, no client in both sets). On that single split the hand rule scored P@20 0.60 / P@50 0.64 and the trees were 0.50-0.55 / 0.60-0.62. The deeper trees did **not** keep their in-sample gain on unseen clients, which is the point of the caveat. The test declining rate is 0.559, so all of these are only modestly above a random pick.
3. **10 different client splits** (depth-2 tree vs hand rule, mean +/- std): P@20 hand 0.69 vs tree 0.59, P@50 hand 0.59 vs tree 0.63, with stds of 0.11-0.18. The hand rule looks better at the very top (K=20), the tree slightly better at K=50, but the spread across splits is as big as the gap, so I would not claim either is clearly better.

**Takeaway:** the in-sample gap does not clearly hold on unseen clients. A depth-2 tree has only 4 distinct scores, so its top-K is mostly ties, which makes this comparison fragile. I would report these as split-dependent results, not a winner.


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.